# 200 — Validate fish-dollars baseline re-evaluation

This notebook is the **gatekeeper** for the transmissivity-uncertainty workflow.

The goal is to prove that we can take the final baseline Pareto-front pumping designs from the original `fish_dollars` PEST++ MOU run, pass those same designs back through the original fish-dollars/PyCap forward model, and reproduce the archived results.

The most important validation for this project is hydrologic:

- total pumping
- depletion at `lpr:total_combined:bdpl`
- streamflow, calculated as `8.6 cfs - depletion`

The fish/ag objective validation is retained as a useful diagnostic because the pumping designs came from the original `fish_dollars` optimization.

## 1. Imports, style settings, and user controls

This section defines the baseline run, caching behavior, validation tolerances, and output folders.

The most important switch is:

```python
RERUN_REEVALUATION = False
```

When `False`, the notebook uses cached re-evaluation results if they already exist. When `True`, it reruns the PyCap/fish-dollars re-evaluation from scratch.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import uncertainty_project_helpers as up

# Apply the project-wide Matplotlib defaults from uncertainty_project_helpers.py.
# You can edit up.PLOT_STYLE in the helper script, or copy it here and customize it.
up.apply_plot_style()

# =============================================================================
# User-editable settings
# =============================================================================

# Baseline fish-dollars PEST++ MOU run to validate.
RUN_NAME = "fish_dollars_baseline_0.0_1.0_0.01"

# Use 5 or 10 for a quick smoke test. Use None for the full final front.
N_TEST_MEMBERS = None

# False = use cached CSV if available. True = force a fresh PyCap re-evaluation.
RERUN_REEVALUATION = False

# Hydrologic definitions used throughout the project.
HISTORIC_STREAMFLOW_CFS = 8.6
DEPLETION_OBS_NAME = "lpr:total_combined:bdpl"

# Main x-axis for hydrologic tradeoff plots.
# For fish_dollars, "effective" pumping is best because the original script shuts
# farms/wells off when the pumping decision is <= 70% of baseline pumping.
PUMPING_COLUMN_FOR_HYDRO_PLOTS = "effective_total_pumping_reeval_cfs"

# =============================================================================
# Validation tolerances
# =============================================================================

# The strict fish-dollars test is intentionally very tight; it may fail due only
# to tiny floating-point/order differences. The practical test is the one we use
# to decide whether the workflow is scientifically equivalent.
STRICT_FISH_PROB_ABS_TOL = 1.0e-8
STRICT_AG_RECEIPTS_REL_TOL = 1.0e-8
PRACTICAL_FISH_PROB_ABS_TOL = 1.0e-6
PRACTICAL_AG_RECEIPTS_REL_TOL = 1.0e-5

# Hydrologic outputs should reproduce almost exactly because these are direct
# PyCap outputs saved in the original obs_pop files.
TOTAL_PUMPING_ABS_TOL_CFS = 1.0e-12
DEPLETION_ABS_TOL_CFS = 1.0e-8
STREAMFLOW_ABS_TOL_CFS = 1.0e-8

# =============================================================================
# Robust project paths
# =============================================================================

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = up.find_lpr_pycap_opt_dir(NOTEBOOK_DIR)
dirs = up.make_project_dirs(NOTEBOOK_DIR, "200", "validate_fish_dollars_baseline_reevaluation")
OUTPUT_DIR = dirs["output_dir"]
CACHE_DIR = dirs["cache_dir"]

CACHE_FILE = CACHE_DIR / "200_baseline_reevaluation_comparison.csv"
ERROR_CACHE_FILE = CACHE_DIR / "200_baseline_reevaluation_errors.csv"

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CACHE_FILE:", CACHE_FILE)

NOTEBOOK_DIR: /workspaces/LPR_redux/LPR_pycap_opt/notebooks/Uncertainty_Project
PROJECT_DIR: /workspaces/LPR_redux/LPR_pycap_opt
OUTPUT_DIR: /workspaces/LPR_redux/LPR_pycap_opt/notebooks/Uncertainty_Project/project_output/200_validate_fish_dollars_baseline_reevaluation
CACHE_FILE: /workspaces/LPR_redux/LPR_pycap_opt/notebooks/Uncertainty_Project/cached_reevaluations/200_baseline_reevaluation_comparison.csv


---
## 2. Load the original model context and the baseline Pareto designs

This section loads the original source-of-truth fish-dollars script from the baseline run folder, reads the Pareto archive, selects the final feasible front-1 members, and loads the saved decision-variable and observation-population files.

The decision-variable population contains the pumping designs. The observation-population files contain archived model outputs, including depletion.

In [5]:
# Load everything needed to call the original fish-dollars forward model.
# The context includes the original model dictionary, fish curve, receipts table,
# observation names, baseline T value, and the script path used for re-evaluation.
context = up.load_fish_dollars_context(RUN_NAME, PROJECT_DIR)

# Read the original PEST++ MOU archive and select the final feasible Pareto-front
# members. These are the management designs we are validating.
pareto_archive = up.read_pareto_archive_summary(RUN_NAME, context.run_dir)
pareto_final = up.select_final_feasible_front1(pareto_archive, n_test_members=N_TEST_MEMBERS)

# Load the saved pumping decision variables and archived observations.
dv_df = up.load_decision_variable_population(context.run_dir)
obs_df = up.load_observation_population(context.run_dir)
q_cols = up.get_q_columns(dv_df)

missing_members = sorted(set(pareto_final["member"]) - set(dv_df.index.astype(str)))

print("Using script:", context.script_path)
print("Run directory:", context.run_dir)
print("Pareto archive rows:", len(pareto_archive))
print("Final feasible front-1 members:", len(pareto_final))
print("Final generation:", pareto_final["generation"].max())
print("Decision-variable population rows:", len(dv_df))
print("Observation-population rows:", 0 if obs_df is None else len(obs_df))
print("Q columns:", len(q_cols))
print("Baseline T:", context.base_T)
print("Reference flow from fish-dollars script:", context.ref_flow)
print("Missing final members in dv_df:", len(missing_members))

if missing_members:
    raise KeyError(f"Missing Pareto members in dv_pop files. First few: {missing_members[:10]}")

Using script: /workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/run_fish_dollars_baseline_0.0_1.0_0.01/run_pycap_standalone_opt_mou_fish_dollars.py
Run directory: /workspaces/LPR_redux/LPR_pycap_opt/pycap_runs/pycap_pest/run_fish_dollars_baseline_0.0_1.0_0.01
Pareto archive rows: 5669
Final feasible front-1 members: 174
Final generation: 50
Decision-variable population rows: 6528
Observation-population rows: 6528
Q columns: 327
Baseline T: 1700.0
Reference flow from fish-dollars script: 8.6
Missing final members in dv_df: 0


---
## 3. Re-evaluate the baseline designs, or load cached re-evaluation results

For each selected Pareto member, the notebook:

1. pulls the pumping design from the archived decision-variable population,
2. applies the same fish-dollars 70% pumping cutoff used by the original script,
3. reruns the original fish-dollars/PyCap forward model with baseline T,
4. saves archived and re-evaluated values side-by-side.

The cache keeps us from having to rerun the same validation every time.

In [6]:
if CACHE_FILE.exists() and not RERUN_REEVALUATION:
    print("Loading cached baseline re-evaluation:", CACHE_FILE)
    comparison = pd.read_csv(CACHE_FILE)
    error_df = pd.read_csv(ERROR_CACHE_FILE) if ERROR_CACHE_FILE.exists() else pd.DataFrame()
else:
    records = []
    errors = []

    for i, row in pareto_final.reset_index(drop=True).iterrows():
        member = str(row["member"])

        # Progress printout keeps long re-evaluations from feeling like they froze.
        if i == 0 or (i + 1) % 25 == 0 or (i + 1) == len(pareto_final):
            print(f"Re-evaluating member {i + 1} of {len(pareto_final)}: {member}")

        q_row = dv_df.loc[member, q_cols].copy()

        try:
            # This is the key validation call: same pumping design, same baseline T,
            # original fish-dollars/PyCap forward model.
            reeval = up.run_original_fish_dollars_get_results(q_row, context, t_factor=1.0)

            # The original fish-dollars script applies a farm/well shutoff rule.
            # Effective pumping is the pumping that actually matters for the
            # hydrologic tradeoff plot.
            q_eff = up.effective_q_after_fish_dollars_cutoff(q_row, context.receipts)

            # Archived pumping is computed from the archived decision variables.
            # Re-evaluated pumping is the same design passed into the forward model.
            # The difference should therefore be zero. We still include it in the
            # summary as a useful sanity check.
            raw_total_pumping_archive_gpm = q_row.sum()
            raw_total_pumping_reeval_gpm = q_row.sum()
            effective_total_pumping_archive_gpm = q_eff.sum()
            effective_total_pumping_reeval_gpm = q_eff.sum()

            raw_total_pumping_archive_cfs = raw_total_pumping_archive_gpm * up.GPM2CFS
            raw_total_pumping_reeval_cfs = raw_total_pumping_reeval_gpm * up.GPM2CFS
            effective_total_pumping_archive_cfs = effective_total_pumping_archive_gpm * up.GPM2CFS
            effective_total_pumping_reeval_cfs = effective_total_pumping_reeval_gpm * up.GPM2CFS

            # Archived hydrologic output from the original MOU obs_pop files.
            depletion_archive_cfs = up.get_obs_value(obs_df, member, DEPLETION_OBS_NAME)
            streamflow_archive_cfs = (
                HISTORIC_STREAMFLOW_CFS - depletion_archive_cfs
                if pd.notna(depletion_archive_cfs)
                else np.nan
            )

            # Re-evaluated hydrologic output from the fresh forward-model call.
            depletion_reeval_cfs = reeval.loc[DEPLETION_OBS_NAME] if DEPLETION_OBS_NAME in reeval.index else np.nan
            streamflow_reeval_cfs = HISTORIC_STREAMFLOW_CFS - depletion_reeval_cfs

            records.append({
                "member": member,
                "generation": row["generation"],

                # Original fish-dollars objectives from the archive.
                "ag_receipts_archive": row["ag_receipts"],
                "fish_prob_archive": row["fish_prob"],

                # Fish-dollars objectives from the re-evaluation.
                "ag_receipts_reeval": reeval.loc["ag_receipts"],
                "fish_prob_reeval": reeval.loc["fish_prob"],

                # Raw pumping totals before the 70% cutoff.
                "raw_total_pumping_archive_gpm": raw_total_pumping_archive_gpm,
                "raw_total_pumping_reeval_gpm": raw_total_pumping_reeval_gpm,
                "raw_total_pumping_archive_cfs": raw_total_pumping_archive_cfs,
                "raw_total_pumping_reeval_cfs": raw_total_pumping_reeval_cfs,

                # Effective pumping totals after the 70% cutoff.
                "effective_total_pumping_archive_gpm": effective_total_pumping_archive_gpm,
                "effective_total_pumping_reeval_gpm": effective_total_pumping_reeval_gpm,
                "effective_total_pumping_archive_cfs": effective_total_pumping_archive_cfs,
                "effective_total_pumping_reeval_cfs": effective_total_pumping_reeval_cfs,

                # Hydrologic validation outputs.
                "depletion_archive_cfs": depletion_archive_cfs,
                "streamflow_archive_cfs": streamflow_archive_cfs,
                "depletion_reeval_cfs": depletion_reeval_cfs,
                "streamflow_reeval_cfs": streamflow_reeval_cfs,
            })

        except Exception as err:
            import traceback
            errors.append({
                "member": member,
                "error": repr(err),
                "traceback": traceback.format_exc(),
            })

    comparison = pd.DataFrame(records)
    error_df = pd.DataFrame(errors)

    comparison.to_csv(CACHE_FILE, index=False)
    if not error_df.empty:
        error_df.to_csv(ERROR_CACHE_FILE, index=False)
    print("Saved cache:", CACHE_FILE)

# -----------------------------------------------------------------------------
# Backward-compatible cache cleanup
# -----------------------------------------------------------------------------
# Earlier versions of this notebook used a single effective_total_pumping_cfs
# column. This block lets the updated notebook still run if it loads an older
# cached CSV.
if "raw_total_pumping_archive_cfs" not in comparison.columns and "raw_total_pumping_cfs" in comparison.columns:
    comparison["raw_total_pumping_archive_cfs"] = comparison["raw_total_pumping_cfs"]
if "raw_total_pumping_reeval_cfs" not in comparison.columns and "raw_total_pumping_archive_cfs" in comparison.columns:
    comparison["raw_total_pumping_reeval_cfs"] = comparison["raw_total_pumping_archive_cfs"]

if "effective_total_pumping_archive_cfs" not in comparison.columns and "effective_total_pumping_cfs" in comparison.columns:
    comparison["effective_total_pumping_archive_cfs"] = comparison["effective_total_pumping_cfs"]
if "effective_total_pumping_reeval_cfs" not in comparison.columns and "effective_total_pumping_archive_cfs" in comparison.columns:
    comparison["effective_total_pumping_reeval_cfs"] = comparison["effective_total_pumping_archive_cfs"]

comparison.head()

Re-evaluating member 1 of 174: gen=47_member=5907_pso
Re-evaluating member 25 of 174: gen=45_member=5670_pso
Re-evaluating member 50 of 174: gen=41_member=5214_pso
Re-evaluating member 75 of 174: gen=32_member=3978_pso
Re-evaluating member 100 of 174: gen=47_member=5999_pso
Re-evaluating member 125 of 174: gen=43_member=5440_pso
Re-evaluating member 150 of 174: gen=34_member=4241_pso
Re-evaluating member 174 of 174: gen=50_member=6385_pso
Saved cache: /workspaces/LPR_redux/LPR_pycap_opt/notebooks/Uncertainty_Project/cached_reevaluations/200_baseline_reevaluation_comparison.csv


,member,generation,ag_receipts_archive,fish_prob_archive,ag_receipts_reeval,fish_prob_reeval,raw_total_pumping_archive_gpm,raw_total_pumping_reeval_gpm,raw_total_pumping_archive_cfs,raw_total_pumping_reeval_cfs,effective_total_pumping_archive_gpm,effective_total_pumping_reeval_gpm,effective_total_pumping_archive_cfs,effective_total_pumping_reeval_cfs,depletion_archive_cfs,streamflow_archive_cfs,depletion_reeval_cfs,streamflow_reeval_cfs
0,gen=47_member=5907_pso,50,9738960.0,0.719193,9.738960e+06,0.719193,42427.949226,42427.949226,94.529471,94.529471,41736.549128,41736.549128,92.989031,92.989031,1.424825,7.175175,1.424825,7.175175
1,56,50,12446400.0,0.533346,1.244640e+07,0.533346,52637.152842,52637.152842,117.275577,117.275577,52637.152842,52637.152842,117.275577,117.275577,4.797320,3.802680,4.797320,3.802680
2,64,50,12377800.0,0.567550,1.237783e+07,0.567550,52241.562957,52241.562957,116.394202,116.394202,52241.562957,52241.562957,116.394202,116.394202,4.457265,4.142735,4.457265,4.142735
3,gen=40_member=5105_pso,50,12283000.0,0.583854,1.228305e+07,0.583854,51868.902507,51868.902507,115.563915,115.563915,51444.954579,51444.954579,114.619359,114.619359,4.281098,4.318902,4.281098,4.318902
4,30,50,12433400.0,0.544694,1.243337e+07,0.544694,52187.234035,52187.234035,116.273157,116.273157,52187.234035,52187.234035,116.273157,116.273157,4.690662,3.909338,4.690662,3.909338


---
## 4. Build the validation summary table

This section calculates the differences between archived and re-evaluated values.

The summary table now includes total pumping difference as an explicit validation check. The mean pumping/depletion/streamflow values were removed from this notebook because this notebook is about **validation**, not final interpretation.

In [7]:
if comparison.empty:
    raise RuntimeError("No members were successfully re-evaluated.")

# =============================================================================
# Fish-dollars objective differences
# =============================================================================
comparison["ag_receipts_diff"] = comparison["ag_receipts_reeval"] - comparison["ag_receipts_archive"]
comparison["fish_prob_diff"] = comparison["fish_prob_reeval"] - comparison["fish_prob_archive"]
comparison["ag_receipts_abs_diff"] = comparison["ag_receipts_diff"].abs()
comparison["fish_prob_abs_diff"] = comparison["fish_prob_diff"].abs()
comparison["ag_receipts_rel_diff"] = (
    comparison["ag_receipts_abs_diff"]
    / comparison["ag_receipts_archive"].abs().replace(0, np.nan)
)

max_ag_abs = comparison["ag_receipts_abs_diff"].max()
max_ag_rel = comparison["ag_receipts_rel_diff"].max()
max_fish_abs = comparison["fish_prob_abs_diff"].max()

strict_fish_dollars_validation_passed = bool(
    (max_ag_rel <= STRICT_AG_RECEIPTS_REL_TOL)
    and (max_fish_abs <= STRICT_FISH_PROB_ABS_TOL)
    and error_df.empty
)

practical_fish_dollars_validation_passed = bool(
    (max_ag_rel <= PRACTICAL_AG_RECEIPTS_REL_TOL)
    and (max_fish_abs <= PRACTICAL_FISH_PROB_ABS_TOL)
    and error_df.empty
)

# =============================================================================
# Pumping validation differences
# =============================================================================
# The re-evaluation should use exactly the same pumping design as the archived
# Pareto member. These differences should be zero or effectively zero.
comparison["raw_total_pumping_diff_cfs"] = (
    comparison["raw_total_pumping_reeval_cfs"]
    - comparison["raw_total_pumping_archive_cfs"]
)
comparison["effective_total_pumping_diff_cfs"] = (
    comparison["effective_total_pumping_reeval_cfs"]
    - comparison["effective_total_pumping_archive_cfs"]
)

max_raw_total_pumping_abs_diff = comparison["raw_total_pumping_diff_cfs"].abs().max()
max_effective_total_pumping_abs_diff = comparison["effective_total_pumping_diff_cfs"].abs().max()
total_pumping_validation_passed = bool(
    (max_effective_total_pumping_abs_diff <= TOTAL_PUMPING_ABS_TOL_CFS)
    and error_df.empty
)

# =============================================================================
# Hydrologic validation differences
# =============================================================================
has_archive_depletion = comparison["depletion_archive_cfs"].notna().any()

if has_archive_depletion:
    comparison["depletion_diff_cfs"] = comparison["depletion_reeval_cfs"] - comparison["depletion_archive_cfs"]
    comparison["streamflow_diff_cfs"] = comparison["streamflow_reeval_cfs"] - comparison["streamflow_archive_cfs"]
    max_depletion_abs_diff = comparison["depletion_diff_cfs"].abs().max()
    max_streamflow_abs_diff = comparison["streamflow_diff_cfs"].abs().max()
else:
    max_depletion_abs_diff = np.nan
    max_streamflow_abs_diff = np.nan

hydrologic_validation_passed = bool(
    has_archive_depletion
    and (max_depletion_abs_diff <= DEPLETION_ABS_TOL_CFS)
    and (max_streamflow_abs_diff <= STREAMFLOW_ABS_TOL_CFS)
    and total_pumping_validation_passed
    and error_df.empty
)

summary = pd.DataFrame({
    "metric": [
        "n_members_attempted",
        "n_members_successful",
        "n_members_error",
        "max_ag_receipts_abs_diff",
        "max_ag_receipts_rel_diff",
        "max_fish_prob_abs_diff",
        "strict_fish_dollars_validation_passed",
        "practical_fish_dollars_validation_passed",
        "max_raw_total_pumping_abs_diff_cfs",
        "max_effective_total_pumping_abs_diff_cfs",
        "total_pumping_validation_passed",
        "historic_streamflow_cfs",
        "depletion_obs_name",
        "has_archive_depletion_values",
        "max_depletion_abs_diff_cfs",
        "max_streamflow_abs_diff_cfs",
        "hydrologic_validation_passed",
    ],
    "value": [
        len(pareto_final),
        len(comparison),
        len(error_df),
        max_ag_abs,
        max_ag_rel,
        max_fish_abs,
        strict_fish_dollars_validation_passed,
        practical_fish_dollars_validation_passed,
        max_raw_total_pumping_abs_diff,
        max_effective_total_pumping_abs_diff,
        total_pumping_validation_passed,
        HISTORIC_STREAMFLOW_CFS,
        DEPLETION_OBS_NAME,
        has_archive_depletion,
        max_depletion_abs_diff,
        max_streamflow_abs_diff,
        hydrologic_validation_passed,
    ]
})

print("Practical fish-dollars validation passed:", practical_fish_dollars_validation_passed)
print("Total pumping validation passed:", total_pumping_validation_passed)
print("Hydrologic validation passed:", hydrologic_validation_passed)
print("Archived depletion values found:", has_archive_depletion)
print("Max effective total pumping absolute difference:", max_effective_total_pumping_abs_diff)
print("Max depletion absolute difference:", max_depletion_abs_diff)
print("Max streamflow absolute difference:", max_streamflow_abs_diff)

display(summary)

Practical fish-dollars validation passed: True
Total pumping validation passed: True
Hydrologic validation passed: True
Archived depletion values found: True
Max effective total pumping absolute difference: 0.0
Max depletion absolute difference: 8.881784197001252e-16
Max streamflow absolute difference: 1.7763568394002505e-15


,metric,value
0,n_members_attempted,174
1,n_members_successful,174
2,n_members_error,0
3,max_ag_receipts_abs_diff,49.649861
4,max_ag_receipts_rel_diff,0.000005
5,max_fish_prob_abs_diff,0.0
6,strict_fish_dollars_validation_passed,False
7,practical_fish_dollars_validation_passed,True
8,max_raw_total_pumping_abs_diff_cfs,0.0
9,max_effective_total_pumping_abs_diff_cfs,0.0


---
## 5. Create validation figures

This section creates two main figures:

1. a single 2x2 validation figure with the four one-to-one checks, and
2. a Pareto-front overlay comparing the original archived hydrologic front with the re-evaluated hydrologic front.

The previous standalone `validated_baseline_streamflow_vs_pumping` plot has been removed because the overlay plot is more useful.

In [11]:
# =============================================================================
# 2x2 validation figure
# =============================================================================
# Each panel compares an archived value to the re-evaluated value. Points should
# fall on the dashed 1:1 line if the re-evaluation is reproducing the original run.
validation_specs = [
    {
        "xcol": "ag_receipts_archive",
        "ycol": "ag_receipts_reeval",
        "title": "Ag receipts",
        "xlabel": "Archived ag receipts",
        "ylabel": "Re-evaluated ag receipts",
    },
    {
        "xcol": "fish_prob_archive",
        "ycol": "fish_prob_reeval",
        "title": "Fish probability",
        "xlabel": "Archived fish probability",
        "ylabel": "Re-evaluated fish probability",
    },
    {
        "xcol": "depletion_archive_cfs",
        "ycol": "depletion_reeval_cfs",
        "title": "Depletion",
        "xlabel": "Archived depletion (cfs)",
        "ylabel": "Re-evaluated depletion (cfs)",
    },
    {
        "xcol": "streamflow_archive_cfs",
        "ycol": "streamflow_reeval_cfs",
        "title": "Streamflow",
        "xlabel": "Archived streamflow (cfs)",
        "ylabel": "Re-evaluated streamflow (cfs)",
    },
]

up.plot_validation_grid_2x2(
    comparison,
    validation_specs,
    OUTPUT_DIR / "200_validation_one_to_one_2x2.png",
    figure_title="Baseline validation: archived vs. re-evaluated outputs",
)

# =============================================================================
# Original archived Pareto front vs. re-evaluated Pareto front
# =============================================================================
# This is the main visual test for the hydrologic tradeoff curve. The original
# front uses archived obs_pop depletion/streamflow values. The re-evaluated front
# uses the fresh forward-model outputs for the exact same pumping designs.
up.plot_archive_vs_reevaluated_front(
    comparison,
    OUTPUT_DIR / "200_original_vs_reevaluated_pareto_front_streamflow.png",
    x_archive="effective_total_pumping_archive_cfs",
    y_archive="streamflow_archive_cfs",
    x_reeval="effective_total_pumping_reeval_cfs",
    y_reeval="streamflow_reeval_cfs",
    xlabel="Effective total pumping after fish-dollars cutoff (cfs)",
    ylabel="Streamflow = 8.6 cfs - depletion (cfs)",
    title="Original archived front vs. re-evaluated front",
)

AttributeError: module 'uncertainty_project_helpers' has no attribute 'plot_validation_grid_2x2'

---
## 6. Save final notebook outputs

This final section writes the comparison table, validation summary, and any error table to the notebook output folder.

In [ ]:
comparison.to_csv(OUTPUT_DIR / "200_baseline_reevaluation_comparison.csv", index=False)
summary.to_csv(OUTPUT_DIR / "200_baseline_reevaluation_validation_summary.csv", index=False)

if not error_df.empty:
    error_df.to_csv(OUTPUT_DIR / "200_baseline_reevaluation_errors.csv", index=False)

print("Saved files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(" -", f.name)